# 0.0 Graph Sanity Check

**Learning:**
- [0.0 Getting Started](../../../Learning/LangGraph/00_foundations/0.0_getting_started.md)
- [1.1 Mental Model](../../../Learning/LangGraph/00_foundations/1.1_mental_model.md)

**Goal:** Confirm LangGraph is installed, build your first `StateGraph`, and compare it side-by-side with an LCEL pipeline.

## Setup

In [4]:
import sys
from pathlib import Path

# Find repo root (directory that contains src/ and requirements.txt)
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

# Returns True if .env file was found; False if not (Docker may already inject vars via env_file)
loaded = load_dotenv(ROOT / ".env")
print(f"Project root: {ROOT}")
print(f"load_dotenv: {loaded}  (.env path: {ROOT / '.env'})")

Project root: /app
load_dotenv: True  (.env path: /app/.env)


## 1. Verify LangGraph Installation

In [5]:
from importlib.metadata import version, PackageNotFoundError
from langgraph.graph import StateGraph, START, END

try:
    print(f"LangGraph version: {version('langgraph')}")
except PackageNotFoundError:
    print("LangGraph version: (installed, version metadata unavailable)")

print("Imports OK: StateGraph, START, END")

LangGraph version: 0.2.53
Imports OK: StateGraph, START, END


## 2. Your First StateGraph

A graph has three parts:
1. **State** — typed data shared between nodes
2. **Nodes** — functions that read and update state
3. **Edges** — connections that define execution order

Flow: `START → greet → END`

In [12]:
from typing import TypedDict  # TypedDict = dict with named, typed fields for graph state


class State(TypedDict):
    message: str  # the only field in our graph state: a string message


def greet(state: State) -> State:
    # Node function: receives full state, returns a partial update
    return {"message": f"Hello, {state['message']}!"}
    # Read input message, prepend "Hello, ", write back to state


graph = StateGraph(State)
# Create an empty graph builder; all nodes share the State schema

graph.add_node("greet", greet)
# Register a node named "greet" that runs the greet() function

graph.add_edge(START, "greet")
# When the graph starts, always go to the "greet" node first

graph.add_edge("greet", END)
# After "greet" finishes, stop the graph (no more nodes)

app = graph.compile()
# Freeze the graph into a runnable application

result = app.invoke({"message": "LangGraph"})
# Run once with initial state; graph executes START → greet → END

result
# Display final state: {'message': 'Hello, LangGraph!'}

{'message': 'Hello, LangGraph!'}

## 3. LCEL vs. StateGraph (Same Transform, Different Model)

| LCEL | LangGraph |
|---|---|
| Linear pipeline | Explicit nodes and edges |
| Great for stateless transforms | Great when steps branch, loop, or persist |

Both approaches below apply the same greeting transform.

In [7]:
from langchain_core.runnables import RunnableLambda

# LCEL: pipe-style transformation
lcel_chain = RunnableLambda(lambda x: {"message": f"Hello, {x['message']}!"})
lcel_result = lcel_chain.invoke({"message": "LangGraph"})

print("LCEL result:", lcel_result)
print("Graph result:", result)
print("Same output:", lcel_result == result)

LCEL result: {'message': 'Hello, LangGraph!'}
Graph result: {'message': 'Hello, LangGraph!'}
Same output: True


## 4. Optional — LCEL with LLM vs. Graph with LLM Node

Skip this cell if `OPENAI_API_KEY` is not configured. It shows how an LLM fits **inside** a graph node—the pattern used throughout this track.

In [15]:
# --- Imports ---
from langchain_core.prompts import ChatPromptTemplate   # reusable prompt template with {variables}
from langchain_core.output_parsers import StrOutputParser  # converts LLM response object → plain string
from src.config import OPENAI_API_KEY                   # reads OPENAI_API_KEY from .env / Docker env
from src.llms.openai_chat import make_openai_chat       # project helper that returns a configured ChatOpenAI

# --- Guard: skip if no API key ---
if not OPENAI_API_KEY:
    # .env missing or empty — avoid a runtime auth error
    print("Skipping LLM demo — set OPENAI_API_KEY in .env to run this cell.")
else:
    # --- Part A: LCEL pipeline (linear) ---

    llm = make_openai_chat()
    # Creates ChatOpenAI(model=..., temperature=0) using credentials from .env

    prompt = ChatPromptTemplate.from_template(
        "Say hello to {name} in one short sentence."
    )
    # Template with one variable `{name}` — filled at invoke time

    lcel_llm_chain = prompt | llm | StrOutputParser()
    # LCEL pipe:  input dict → prompt formats text → llm generates → parser extracts string
    # Equivalent flow: {"name": "..."} → "Say hello to ..." → AIMessage → "Hello, ..."

    lcel_reply = lcel_llm_chain.invoke({"name": "LangGraph"})
    # Run the chain once; pass the variable value for `{name}`

    print("LCEL + LLM:", lcel_reply)
    # Print the final plain-text string from the LCEL pipeline

    # --- Part B: LangGraph (same LLM logic wrapped as a node) ---

    class LlmState(TypedDict):
        name: str    # input: who to greet
        reply: str   # output: LLM response stored back into graph state

    def llm_node(state: LlmState) -> LlmState:
        # Node function: reads current state, returns a partial state update
        reply = lcel_llm_chain.invoke({"name": state["name"]})
        # Reuse the same LCEL chain inside the node
        return {"reply": reply}
        # LangGraph merges {"reply": ...} into the existing state (keeps `name`)

    llm_graph = StateGraph(LlmState)
    # Create a graph whose shared data shape is LlmState

    llm_graph.add_node("llm", llm_node)
    # Register one node named "llm" that runs llm_node()

    llm_graph.add_edge(START, "llm")
    # Entry point: when graph runs, go to the "llm" node first

    llm_graph.add_edge("llm", END)
    # After "llm" finishes, stop the graph

    llm_app = llm_graph.compile()
    # Turn the graph definition into a runnable app (like chain.compile())

    graph_reply = llm_app.invoke({"name": "LangGraph", "reply": ""})
    # Run graph once with initial state; `reply` starts empty and gets filled by the node

    print("Graph + LLM:", graph_reply["reply"])
    # Print the reply from final graph state — should match the LCEL result above

LCEL + LLM: Hello, LangGraph! Excited to connect and explore together!
Graph + LLM: LangGraph Hello, LangGraph! Excited to connect and explore together!


## Exit Criteria Checklist

- [ ] LangGraph imports successfully
- [ ] You ran `START → greet → END` and got a transformed message
- [ ] You can explain why a graph is better than a `while` loop for agent control
- [ ] You see that LCEL and LangGraph solve different problems—and work together

**Next:** [1.1 First StateGraph](../../../Learning/LangGraph/01_beginner/1.1_first_stategraph.md) → `../01_beginner/1.1_first_stategraph.ipynb`